# 01 - Data Preprocessing Basics

Sprint 5, Notebook 1. Continuing with the same telecom churn dataset I used in Sprint 4 for EDA. Before jumping into cleaning code, this notebook is just about getting the concepts straight - what preprocessing actually is, why it matters, and the vocabulary I'll keep using for the rest of the sprint (data quality, leakage, train/val/test, etc).

## 1. What is Data Preprocessing, and why bother

Raw data almost never comes in a shape a model can use directly. There will be missing cells, wrong data types, duplicate rows, values that don't make sense (age of 200), and categories written five different ways. Preprocessing is the set of steps I take to turn that raw dump into something a model can actually learn from - fixing types, filling or dropping missing values, removing junk rows, scaling numbers, encoding text categories into numbers, and so on.

Why does it matter so much? Because a model doesn't know the difference between real signal and garbage - it will just fit to whatever you give it. If `TotalCharges` is stored as text because of 15 blank entries, the column can't even be used in a calculation until that's fixed. If `MonthlyCharges` has a few rows sitting at 500+ next to a normal range of 20-100, distance-based models like KNN or anything using Euclidean distance will get dragged around by those few rows. So the model quality has a hard ceiling set by the data quality - no amount of tuning fixes bad input.

### What is Data Preprocessing?

Data preprocessing means cleaning and preparing raw data before feeding it 
to a machine learning model. Raw data usually has missing values, wrong 
data types, duplicates, and outliers - preprocessing fixes these.

**Example:** A "Salary" column stored as text "50,000" cannot be used 
directly in a model - it must be converted to a number first.

In [11]:
import pandas as pd

df = pd.read_csv("customer_churn.csv")
print("Dataset loaded. Shape:", df.shape)
df.head()

Dataset loaded. Shape: (5060, 14)


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,InternetService,TechSupport,SeniorCitizen,Partner,PaperlessBilling,Churn
0,CUST00001,56.0,male,5,23.68,7990.55,Two Year,Mailed Check,No,Yes,1,Yes,Yes,Yes
1,CUST00002,69.0,Male,9,127.71,NaN,One Year,Bank Transfer,No,No,1,Yes,No,No
2,CUST00003,46.0,M,9,122.48,6140.18,One Year,Credit Card,No,No,1,No,No,No
3,CUST00004,32.0,male,13,53.67,NaN,Month-to-Month,Electronic Check,Fiber Optic,Yes,0,Yes,No,Yes
4,CUST00005,60.0,male,10,112.13,5151.27,Month-to-Month,Mailed Check,Fiber Optic,Yes,1,Yes,Yes,Yes


### Why is Data Preprocessing required?

Real-world data is never perfectly clean. If we skip preprocessing:
- Models may crash due to wrong data types.
- Models may learn wrong patterns from bad/missing values.
- Predictions become unreliable.

**Example:** If "TotalCharges" has blank values, a model will throw an 
error when trying to calculate with it.

In [12]:
# Check if any column has issues that would break a model
print(df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

CustomerID           object
Age                 float64
Gender               object
TenureYears           int64
MonthlyCharges      float64
TotalCharges        float64
ContractType         object
PaymentMethod        object
InternetService      object
TechSupport          object
SeniorCitizen         int64
Partner              object
PaperlessBilling     object
Churn                object
dtype: object

Missing values:
 CustomerID            0
Age                  80
Gender                0
TenureYears           0
MonthlyCharges        0
TotalCharges        151
ContractType          0
PaymentMethod         0
InternetService       0
TechSupport           0
SeniorCitizen         0
Partner               0
PaperlessBilling      0
Churn                 0
dtype: int64


### Raw Data vs Clean Data

**Raw Data:** missing values, wrong types, duplicates, outliers, 
inconsistent categories (e.g. "Male", "male", "M").

**Clean Data:** no missing values (or properly handled), correct types, 
no duplicates, outliers treated, consistent category values.

In [13]:
# Example: inconsistent category values in a column
if "gender" in df.columns:
    print(df["gender"].unique())
else:
    print(df["ContractType"].unique())

['Two Year' 'One Year' 'Month-to-Month']


### Data Quality

Data quality is judged mainly on 4 things:
- **Completeness** - are values missing?
- **Consistency** - is the same info written the same way everywhere?
- **Accuracy** - does data reflect the real value?
- **Validity** - does data follow expected rules (e.g. age can't be -5)?

In [14]:
# Quick data quality snapshot
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNegative values in MonthlyCharges:", (df["MonthlyCharges"] < 0).sum())

Missing values:
 CustomerID            0
Age                  80
Gender                0
TenureYears           0
MonthlyCharges        0
TotalCharges        151
ContractType          0
PaymentMethod         0
InternetService       0
TechSupport           0
SeniorCitizen         0
Partner               0
PaperlessBilling      0
Churn                 0
dtype: int64

Duplicate rows: 40

Negative values in MonthlyCharges: 0


### Data Consistency

Data is consistent when the same type of information is written in the 
same format across the whole dataset.

**Example:** "Yes"/"No" should not appear as "yes", "Y", "1" in some rows.

In [15]:
# Check for inconsistent text formatting in Churn column
print(df["Churn"].unique())

['Yes' 'No']


### Data Completeness

Completeness means how much of the required data is actually present, 
without missing values.

**Example:** If 30% of "TotalCharges" is missing, the dataset is 
incomplete for that column.

In [16]:
# Percentage of missing values per column
missing_percent = (df.isnull().sum() / len(df)) * 100
print(missing_percent)

CustomerID          0.000000
Age                 1.581028
Gender              0.000000
TenureYears         0.000000
MonthlyCharges      0.000000
TotalCharges        2.984190
ContractType        0.000000
PaymentMethod       0.000000
InternetService     0.000000
TechSupport         0.000000
SeniorCitizen       0.000000
Partner             0.000000
PaperlessBilling    0.000000
Churn               0.000000
dtype: float64


### Data Accuracy

Accuracy means the data correctly reflects the real-world value.

**Example:** A customer's age recorded as 150 is clearly inaccurate - 
it does not reflect a real value.

In [17]:
# Check for unrealistic values, e.g. Age column
if "Age" in df.columns:
    print(df["Age"].describe())

count    4980.000000
mean       43.760442
std        15.381362
min        -5.000000
25%        31.000000
50%        44.000000
75%        56.000000
max       150.000000
Name: Age, dtype: float64


### Data Validity

Validity means data follows the rules it is supposed to follow.

**Example:** "MonthlyCharges" should never be negative. "TenureYears" 
should never be more than a customer's total possible tenure.

In [18]:
# Validity check example
invalid_charges = df[df["MonthlyCharges"] < 0]
print("Invalid MonthlyCharges rows:", len(invalid_charges))

Invalid MonthlyCharges rows: 0


### Data Integrity

Data integrity means the data stays correct and unchanged in an unwanted 
way as it moves through different processing steps. Broken integrity leads 
to wrong conclusions.

**Example:** If a merge/join operation accidentally duplicates rows, 
integrity is broken.

In [19]:
# Check row count before and after a simple operation to confirm integrity
before = len(df)
df_check = df.drop_duplicates()
after = len(df_check)
print("Rows before:", before, "| Rows after removing duplicates:", after)

Rows before: 5060 | Rows after removing duplicates: 5020


### Data Leakage

Data leakage happens when information from outside the training data 
(like the test set, or future data) is accidentally used while training. 
This makes the model look very accurate in testing but fail in real use.

**Example:** Using "TotalCharges" (which is calculated from months already 
passed, including future churn month) directly to predict "Churn" could 
leak information.

In [20]:
# Just an illustration - checking correlation to spot possible leakage risk
print(df.corr(numeric_only=True)["MonthlyCharges"])

Age              -0.010361
TenureYears       0.011762
MonthlyCharges    1.000000
TotalCharges     -0.035283
SeniorCitizen    -0.016925
Name: MonthlyCharges, dtype: float64


### Training, Validation and Test Data

- **Training Data** - used to teach the model.
- **Validation Data** - used to tune the model and choose settings.
- **Test Data** - used only at the end to check performance on unseen data.

In [21]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(df, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print("Train size:", train_data.shape)
print("Validation size:", val_data.shape)
print("Test size:", test_data.shape)

Train size: (3542, 14)
Validation size: (759, 14)
Test size: (759, 14)
